In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn as skl
import sqlite3 as sql   
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
print("All libraries imported successfully!")

In [ ]:
# Load dataset
df = pd.read_csv("data/data.csv")

# Basic dataset information
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

# Remove duplicates
df.drop_duplicates(inplace=True)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Fill missing values
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

print("\nRemaining Missing Values:")
print(df.isnull().sum())


In [ ]:
print(df.columns)

categorical_cols = [ 'gender','Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService', 'OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod',]

for col in categorical_cols:
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    churn_rate.plot(kind='bar')
    plt.title(f'Churn Rate by {col}')
    plt.show()

In [ ]:
numerical_cols = [
    
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
for col in numerical_cols:
    plt.figure(figsize=(7,4))
    df.boxplot(column=col, by="Churn")
    plt.title(f"{col} vs Churn")
    plt.suptitle("")
    plt.xlabel("Churn")
    plt.ylabel(col)
    plt.show()

total charges
The median TotalCharges of customers who did not churn is significantly higher than that of customers who churned. This indicates that customers who remain with the company accumulate larger total payments over time. Although a few customers with high TotalCharges still churned (visible as outliers), the majority of churned customers have relatively lower TotalCharges.
tenure
higher tenure less chances to leave

In [ ]:
#analysis of churn
df["Churn"].value_counts().plot(kind='bar')
plt.title('Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.show()      



In [ ]:
for col in df.columns:
    print(col)
    print(df[col].unique())
    print("----------------")

In [ ]:
#droping the attributes which are not required for the analysis
print(df.columns)
df.drop(['customerID'], axis=1, inplace=True)

# Don't drop these yet
# df.drop(['gender'], axis=1, inplace=True)
# df.drop(['PhoneService'], axis=1, inplace=True)
# df.drop(['PaperlessBilling'], axis=1, inplace=True)
# df.drop(['MultipleLines'], axis=1, inplace=True)
# df.drop(['OnlineBackup'], axis=1, inplace=True)
# df.drop(['DeviceProtection'], axis=1, inplace=True)
# df.drop(['StreamingMovies'], axis=1, inplace=True)
# df.drop(['StreamingTV'], axis=1, inplace=True)

print(df.columns)

In [ ]:
X=df.drop('Churn', axis=1)
Y=df['Churn']
#encoding the target into numerical values
le = LabelEncoder()
    
Y = le.fit_transform(Y)
#encoding the categorical attributes into numerical values
X=pd.get_dummies(X, drop_first=True)
print(X.shape)
print(X.head())
print(X.dtypes)



In [ ]:
#logistic regression model


X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)
print("all done")
#scalling the data
scaler = skl.preprocessing.StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("all done")
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, Y_train)
Y_pred = lr.predict(X_test_scaled)
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print(classification_report(Y_test, Y_pred))

In [ ]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, Y_train)

Y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(Y_test, Y_pred_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred_dt))

print("\nClassification Report:")
print(classification_report(Y_test, Y_pred_dt))

In [ ]:


rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42
)

rf.fit(X_train, Y_train)

Y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(Y_test, Y_pred_rf))
print(confusion_matrix(Y_test, Y_pred_rf))
print(classification_report(Y_test, Y_pred_rf))

In [ ]:
import joblib

joblib.dump(lr, "churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("all done")